## 1. Design Backed support for the HR Access-Control Problem

A company's HR system models employees using a class hierarchy rooted at Employee, with Manager as a specialization. The system has three data-protection requirements:

The employee's name must be freely readable and writable from anywhere in the codebase.

The salary must never be overwritten directly by assigning to it from outside the class. It may only change through a method that revises it by a percentage hike between 0 and 50 inclusive; any other value must raise an Error.

The leave_balance must be usable by Employee itself and by any class that extends it, such as Manager, but code outside this class family should not be able to rely on accessing it directly.
Implement Employee and a Manager subclass with apply_leave(days), which reduces leave balance and raises an Error if days exceed the remaining balance.

## Tasks:

Implement the Employee class using appropriate access-control conventions.

Implement the Manager subclass.

Implement salary revision using a percentage hike between 0 and 50 inclusive.

Implement apply_leave(days).

Demonstrate that attaching '_____salary' to an Employee instance from
outside the class does not read the real salary. Explain what Python
actually did to the attribute name.

Create an employee/manager object and perform some salary and leave operations.

## File Handling:
Create a text file named employee_records.txt.

Store the following information in the file:

Employee name

Salary

Remaining leave balance

Store the information in the file.

Read the stored information back and display it.

Properly close the file.

[ Encapsulation+ Text File Handling open(), w, write(), read(), close()]


In [ ]:
class SalaryError(Exception):
    pass
class LeaveError(Exception):
    pass
class Employee:
    def __init__(self, name, salary, leave_balance):
        self.name = name
        self.__salary = salary
        self._leave_balance = leave_balance

    def get_salary(self):
        return self.__salary

    def revise_salary(self, hike_percent):
        if hike_percent < 0 or hike_percent > 50:
            raise SalaryError('hike% must be between 0 and 50, got ' + str(hike_percent))
        self.__salary = self.__salary + self.__salary * hike_percent / 100
        return self.__salary

    def apply_leave(self, days):
        if days > self._leave_balance:
            raise LeaveError('only ' + str(self._leave_balance) + ' leaves left')
        self._leave_balance = self._leave_balance - days
        return self._leave_balance
class Manager(Employee):
    pass
e = Employee('viraj', 50000, 12)
print('name:', e.name)
e.name = 'viraj Patel'
print('new name:', e.name)
print('salary:', e.get_salary())
print('after +10% hike:', e.revise_salary(10))
try:
    e.revise_salary(60)
except SalaryError as err:
    print('SalaryError:', err)
e.__salary = 999999
print('decoy e.__salary   :', e.__salary)
print('real salary        :', e.get_salary())
print('Python stored it as :', e._Employee__salary)
m = Manager('Ravi', 80000, 5)
print('leaves left after 2:', m.apply_leave(2))
try:
    m.apply_leave(10)
except LeaveError as err:
    print('LeaveError:', err)
f = open('employee_records.txt', 'w')
f.write('Name: ' + e.name + '\n')
f.write('Salary: ' + str(e.get_salary()) + '\n')
f.write('Remaining leave: ' + str(e._leave_balance) + '\n')
f.close()
f = open('employee_records.txt', 'r')
print(f.read())
f.close()

: 

## 2. Code in Python for an e-commerce checkout system
needs a common contract for **payment gateways (Credit Card, UPI,** and others added later). The design must make it structurally impossible to create a generic, gateway-less payment object — every concrete gateway must be forced to supply its own payment logic, while still inheriting a shared receipt-formatting method from the common ancestor.

### Tasks:
- Design the **base class** and **two concrete gateways**, [`CreditCardProcessor` and `UPIProcessor`] that satisfy the requirement above.
- Show that attempting to instantiate the base class directly fails, and capture/print the exact exception message.
- Process a payment of `1500` through both concrete gateways and print their receipts.

### File Handling:
- Create a text file named `payment_receipts.txt`.
- Store the receipts generated by both payment processors in the file.
- Use `write()` to store each receipt.
- Read the file using `readlines()` and display all stored receipts.

[Abstraction + Text File Handling - w, write(), readlines(), close()]

In [ ]:
from abc import ABC, abstractmethod

class PaymentGateway(ABC):
    @abstractmethod
    def process_payment(self, amount):
        pass

    def format_receipt(self, amount, method_name):
        return f"Receipt: Successfully processed {amount} using {method_name}."

class CreditCardProcessor(PaymentGateway):
    def process_payment(self, amount):
        return self.format_receipt(amount, "Credit Card")

class UPIProcessor(PaymentGateway):
    def process_payment(self, amount):
        return self.format_receipt(amount, "UPI")

try:
    generic_gateway = PaymentGateway()
except TypeError as e:
    print("TypeError caught when instantiating PaymentGateway:", e)

cc = CreditCardProcessor()
upi = UPIProcessor()

r1 = cc.process_payment(1500)
r2 = upi.process_payment(1500)

print(r1)
print(r2)

f = open('payment_receipts.txt', 'w')
f.write(r1 + '\n')
f.write(r2 + '\n')
f.close()

f = open('payment_receipts.txt', 'r')
lines = f.readlines()
for line in lines:
    print(line.strip())
f.close()

## 3. WAP for A notification service
must send the same logical message through Email, SMS, and Push channels, but each channel formats the message differently: **Email** tags it, **SMS** truncates the message to its first 20 characters, and **Push** uppercases it. Calling code should never need to know which concrete channel it is talking to.

### Tasks:
- Design a common `Notifier` ancestor whose `send(message)` is meant to be replaced by every channel.
- Make it fail loudly if a subclass forgets to supply its own version.
- Implement: `EmailNotifier`, `SMSNotifier`, `PushNotifier`
- Implement the required formatting rules.
- Write a `broadcast(notifiers, message)` function that sends one message through a heterogeneous list using a single loop.
- Run it using: 'Server maintenance scheduled tonight'
- Display the notification generated by each channel.

### File Handling:
- Create a file named `notifications.txt`.
- Store every generated notification in the file.
- Use **append mode (a)** so that new notification messages are added without deleting previously stored notifications.
- Use `write()` to append each notification.
- Read the complete file and display all stored notifications.

[ Method Overriding + File Handling concepts: a, write(), read(), close() ]

In [ ]:
class Notifier:
    def send(self, message):
        raise NotImplementedError("Subclass must implement abstract method")

class EmailNotifier(Notifier):
    def send(self, message):
        return f"[Email] {message}"

class SMSNotifier(Notifier):
    def send(self, message):
        return f"SMS: {message[:20]}"

class PushNotifier(Notifier):
    def send(self, message):
        return f"PUSH: {message.upper()}"

def broadcast(notifiers, message):
    for notifier in notifiers:
        notification = notifier.send(message)
        print(notification)
        f = open('notifications.txt', 'a')
        f.write(notification + '\n')
        f.close()

notifiers = [EmailNotifier(), SMSNotifier(), PushNotifier()]
broadcast(notifiers, 'Server maintenance scheduled tonight')

f = open('notifications.txt', 'r')
print("\nFile contents:")
print(f.read())
f.close()

## 4. WAP for the Calculator class
needs a single method named `add` whose behavior depends on the type of arguments provided. The method should accept two or more arguments:
- For **integers**, all arguments should be summed arithmetically.
- For **strings**, all arguments should be joined using a single space, with unnecessary leading and trailing spaces removed.
- For **lists**, all lists should be merged into a single sorted list containing only unique elements.
- **Passing mismatched types**, such as an integer and a string, should fail clearly.

### Tasks:
- If you simply write `def add(self, a, b):` multiple times with different bodies in the same class, what actually happens in Python? State the rule and its consequence in 1–2 sentences.
- Implement the `Calculator` class with a single `add` method that supports two or more arguments and satisfies all three behaviors described above. Use an appropriate mechanism in Python to implement method overloading-like behavior based on argument types.
- **Test the method with:**
  - `add(3, 4)`
  - `add(3, 4, 5, 6)`
  - `add("hello", "world")`
  - `add("hello ", " world", " python ")`
  - `add([1, 2, 3], [2, 3, 4])`
  - `add([1, 2], [2, 3], [4, 5])`
- Test the method with **mismatched types**: `add(3, "4")`
- The program should produce a clear Validation.
- State one limitation of this approach compared with true compile-time method overloading in languages such as Java or C++.

[Method overloading ]

In [ ]:
# Answer 1: In Python, if you define the same method multiple times in a class, the last definition overwrites all previous ones. As a consequence, only the most recently defined method can be called.

class Calculator:
    def add(self, *args):
        if not args:
            return None
        
        first_type = type(args[0])
        for arg in args:
            if type(arg) != first_type:
                raise TypeError(f"Validation Error: Mismatched types, expected {first_type.__name__}, got {type(arg).__name__}")
        
        if first_type == int:
            return sum(args)
        elif first_type == str:
            return " ".join([arg.strip() for arg in args])
        elif first_type == list:
            merged = []
            for lst in args:
                merged.extend(lst)
            return sorted(list(set(merged)))
        else:
            raise TypeError("Unsupported type")

calc = Calculator()
print("add(3, 4):", calc.add(3, 4))
print("add(3, 4, 5, 6):", calc.add(3, 4, 5, 6))
print('add("hello", "world"):', calc.add("hello", "world"))
print('add("hello ", " world", " python "):', calc.add("hello ", " world", " python "))
print("add([1, 2, 3], [2, 3, 4]):", calc.add([1, 2, 3], [2, 3, 4]))
print("add([1, 2], [2, 3], [4, 5]):", calc.add([1, 2], [2, 3], [4, 5]))

try:
    calc.add(3, "4")
except TypeError as e:
    print(e)

# Answer 2: One limitation compared to Java/C++ is that type mismatch errors are caught at runtime rather than at compile-time, which can lead to unexpected crashes when running the code. It also makes the function body complex.

## 5. A Distance class
is required to represent a distance in meters. Create two or more Distance objects and provide support for performing **arithmetic and comparison operations directly between objects.**

The class should:
- Store the distance value in meters.
- Overload the `+` operator so that two or more Distance objects can be added.
- Overload the `-` operator to find the difference between two Distance objects.
- Overload the `==` operator to check whether two Distance objects represent the same distance.
- Overload the `<` operator to compare two Distance objects.
- Display the distance in a readable format when the object is printed.

### Tasks:
- Create a `Distance` class with a constructor that accepts distance in meters.
  - Overload the `+` operator using `__add__()` so that `d1 + d2` returns a new Distance object containing the sum.
  - Extend the `+` operator so that multiple objects can be added: `d1 + d2 + d3`
  - Overload the `-` operator using `__sub__()`.
  - Overload the `==` operator using `__eq__()`.
  - Overload the `<` operator using `__lt__()`.
  - Overload `__str__()` so that printing an object displays the distance in meters.
- Test the class using at least three objects.
- Explain how Python translates an expression such as `d1 + d2` into a special method call.
- Create a binary file named: `distances.dat`
- Store the `Distance` objects in the binary file using the `pickle` module.
- Write the objects.
- Read the objects back.
- Display the distances read from the binary file.
- Store at least three Distance objects and retrieve them from the file.

[operator overloading +File Handling concepts: Binary file, wb, rb, pickle.dump(), pickle.load() ]

In [ ]:
import pickle

class Distance:
    def __init__(self, meters):
        self.meters = meters

    def __add__(self, other):
        if isinstance(other, Distance):
            return Distance(self.meters + other.meters)
        raise TypeError("Unsupported operand type")

    def __sub__(self, other):
        if isinstance(other, Distance):
            return Distance(abs(self.meters - other.meters))
        raise TypeError("Unsupported operand type")

    def __eq__(self, other):
        if isinstance(other, Distance):
            return self.meters == other.meters
        return False

    def __lt__(self, other):
        if isinstance(other, Distance):
            return self.meters < other.meters
        raise TypeError("Unsupported operand type")

    def __str__(self):
        return f"{self.meters} meters"

d1 = Distance(10)
d2 = Distance(20)
d3 = Distance(5)

print(f"d1: {d1}, d2: {d2}, d3: {d3}")
print("d1 + d2:", d1 + d2)
print("d1 + d2 + d3:", d1 + d2 + d3)
print("d2 - d1:", d2 - d1)
print("d1 == d2:", d1 == d2)
print("d1 < d2:", d1 < d2)

# Answer: Python translates the expression `d1 + d2` into `d1.__add__(d2)`.

distances_list = [d1, d2, d3]

f = open("distances.dat", "wb")
pickle.dump(distances_list, f)
f.close()

f = open("distances.dat", "rb")
loaded_distances = pickle.load(f)
f.close()

print("\nLoaded distances from distances.dat:")
for d in loaded_distances:
    print(d)

## 6. Problem Statement:
A media player needs to play items from `AudioTrack`, `VideoClip`, and `Podcast` — three classes that share no common ancestor and were written by different teams. A `Playlist` should be able to play any object that exposes a callable `play()` method, and should skip (not crash on) anything that doesn't, printing a message instead.

### Tasks:
- Implement `AudioTrack`, `VideoClip`, and `Podcast`, each with its own `play()` returning a distinct message, and confirm none of them share a base class other than Object.
- Implement `Playlist.play_all()` that works correctly across all three types plus a plain string placed in the same list: `not_playable`. Skip the string instead of crashing.
- Print an appropriate message when an object does not provide a callable `play()` method.
- Run `play_all()` on:
  ```python
  [
      AudioTrack(...),
      VideoClip(...),
      Podcast(...),
      "not_playable"
  ]
  ```
- Create a CSV file named: `playlist_history.csv`
- Store the result of each playlist item in the CSV file.
- Use `csv.writer()` to create the writer.
- Use `writerow()` to write each result.
- Include suitable column headings such as: Type, Status, Message
- Run the playlist again and store all results in the CSV file.
- Read the CSV file.
- Display the stored records.

[duck Typing +File Handling concepts: csv, csv.writer(), writerow(), reader(). ]

In [ ]:
import csv

class AudioTrack:
    def play(self):
        return "Playing audio track"

class VideoClip:
    def play(self):
        return "Playing video clip"

class Podcast:
    def play(self):
        return "Playing podcast episode"

class Playlist:
    @staticmethod
    def play_all(items):
        results = []
        for item in items:
            if hasattr(item, 'play') and callable(getattr(item, 'play')):
                msg = item.play()
                print(msg)
                results.append(["Type: " + type(item).__name__, "Played", msg])
            else:
                msg = "Skipping item without callable play() method"
                print(msg)
                results.append(["Type: " + type(item).__name__, "Skipped", msg])
        return results

playlist = [
    AudioTrack(),
    VideoClip(),
    Podcast(),
    "not_playable"
]

results = Playlist.play_all(playlist)

f = open('playlist_history.csv', 'w', newline='')
writer = csv.writer(f)
writer.writerow(["Type", "Status", "Message"])
for row in results:
    writer.writerow(row)
f.close()

print("\nReading from playlist_history.csv:")
f = open('playlist_history.csv', 'r')
reader = csv.reader(f)
for row in reader:
    print(row)
f.close()